In [1]:
import os
import requests
pdf_path="human-nutrition-text.pdf"

if not os.path.exists(pdf_path):
    print("does not exist downloading")

    url="https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf"

    filename=pdf_path

    response=requests.get(url)

    if response.status_code==200:
        with open(filename,"wb") as file:
            file.write(response.content)
            print("file has benn downloded")
    else:
        print("error {response.status_code}")
else:
    print("file {pdf_path} esisits")


file {pdf_path} esisits


In [2]:
import fitz 
from tqdm.auto import  tqdm

def text_formatter(text:str)-> str:
    """ performs minor formatting """

    cleaned_text=text.replace("\n"," ").strip()

    return cleaned_text


def open_and_read_pdf(pdf_path:str)->list[dict]:
    doc=fitz.open(pdf_path)
    pages_and_texts=[]
    for page_number,page in enumerate(tqdm(doc)):
        text=page.get_text()
        text=text_formatter(text=text)
        pages_and_texts.append({"page_number":page_number-41,
                                "page_char_count":len(text),
                                "page_word_count":len(text.split(" ")),
                                "page_token_count":len(text)/4,
                                "text" :text})
    return pages_and_texts


pages_and_text=open_and_read_pdf(pdf_path=pdf_path)
pages_and_text[:2]


  0%|          | 0/1208 [00:00<?, ?it/s]

[{'page_number': -41,
  'page_char_count': 29,
  'page_word_count': 4,
  'page_token_count': 7.25,
  'text': 'Human Nutrition: 2020 Edition'},
 {'page_number': -40,
  'page_char_count': 0,
  'page_word_count': 1,
  'page_token_count': 0.0,
  'text': ''}]

In [5]:
import random

random.sample(pages_and_text,k=3)

[{'page_number': 951,
  'page_char_count': 1274,
  'page_word_count': 239,
  'page_token_count': 318.5,
  'text': 'Image by  Allison  Calabrese /  CC BY 4.0  Physical Activity Duration and Fuel Use  The respiratory system plays a vital role in the uptake and delivery  of oxygen to muscle cells throughout the body. Oxygen is inhaled  by the lungs and transferred from the lungs to the blood where  the cardiovascular system circulates the oxygen-rich blood to the  muscles. \xa0The oxygen is then taken up by the muscles and can be  used to generate ATP. When the body is at rest, the heart and  lungs are able to supply the muscles with adequate amounts of  oxygen to meet the aerobic metabolism energy needs. However,  during physical activity your muscles energy and oxygen needs are  increased. In order to provide more oxygen to the muscle cells, your  heart rate and breathing rate will increase. The amount of oxygen  that is delivered to the tissues via the cardiovascular and respiratory  s

In [10]:
import pandas as pd 
df=pd.DataFrame(pages_and_text)
df.head()


,page_number,page_char_count,page_word_count,page_token_count,text
0,-41,29,4,7.25,Human Nutrition: 2020 Edition
1,-40,0,1,0.00,
2,-39,320,54,80.00,Human Nutrition: 2020 Edition UNIVERSITY OF ...
3,-38,212,32,53.00,Human Nutrition: 2020 Edition by University of...
4,-37,797,145,199.25,Contents Preface University of Hawai‘i at Mā...


In [11]:
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_token_count
count,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,198.30,287.00
std,348.86,560.38,95.76,140.10
min,-41.00,0.00,1.00,0.00
25%,260.75,762.00,134.00,190.50
50%,562.50,1231.50,214.50,307.88
75%,864.25,1603.50,271.00,400.88
max,1166.00,2308.00,429.00,577.00


#splitting pagesinto senrternces

#spacy and nltk

In [16]:
from spacy.lang.en import English

nlp=English()

nlp.add_pipe("sentencizer")

doc=nlp("this is a sentence. this is an another sentence. i like c++")

assert len(list(doc.sents))==3

list(doc.sents)





[this is a sentence., this is an another sentence., i like c++]

In [18]:
for item in tqdm(pages_and_text):
    item["sentences"]=list(nlp(item["text"]).sents)

    item["sentences"]=[str(sentences) for sentences in item["sentences"]]

    item["pages_sentence_count_spacy"]=len(item["sentences"])


  0%|          | 0/1208 [00:00<?, ?it/s]

In [20]:
random.sample(pages_and_text,k=1)

[{'page_number': 624,
  'page_char_count': 1160,
  'page_word_count': 207,
  'page_token_count': 290.0,
  'text': 'at once to get the RDA of calcium.  Dietary Reference Intake for Calcium  The recommended dietary allowances (RDA) for calcium are listed  in Table 10.1 “Dietary Reference Intakes for Calcium”. The RDA is  elevated to 1,300 milligrams per day during adolescence because  this is the life stage with accelerated bone growth. Studies have  shown that a higher intake of calcium during puberty increases  the total amount of bone tissue that accumulates in a person. For  women above age fifty and men older than seventy-one, the RDAs  are also a bit higher for several reasons including that as we age,  calcium absorption in the gut decreases, vitamin D3 activation is  reduced, and maintaining adequate blood levels of calcium is  important to prevent an acceleration of bone tissue loss (especially  during menopause). Currently, the dietary intake of calcium for  females above age n

In [21]:
df=pd.DataFrame(pages_and_text)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_token_count,pages_sentence_count_spacy
count,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,198.30,287.00,10.32
std,348.86,560.38,95.76,140.10,6.30
min,-41.00,0.00,1.00,0.00,0.00
25%,260.75,762.00,134.00,190.50,5.00
50%,562.50,1231.50,214.50,307.88,10.00
75%,864.25,1603.50,271.00,400.88,15.00
max,1166.00,2308.00,429.00,577.00,28.00


### chunkking our sentences together

splitting large chunks into a smaller 

In [27]:
num_sentence_chunk=10

def split_list(input_list:list[str],slice_size:int=num_sentence_chunk)->list[list[str]]:
    return[input_list[i:i+slice_size] for i in range(0,len(input_list),slice_size)]


test_list=list(range(25))

print(split_list(test_list))

[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9], [10, 11, 12, 13, 14, 15, 16, 17, 18, 19], [20, 21, 22, 23, 24]]


In [29]:
#loop through pages and split sentences into chunks 


for item in tqdm(pages_and_text):
    item["sentences_chunks"]=split_list(input_list=item["sentences"],slice_size=num_sentence_chunk)

    item["num_chunks"]=len(item["sentences_chunks"])



  0%|          | 0/1208 [00:00<?, ?it/s]

In [36]:
random.sample(pages_and_text,k=1)

[{'page_number': 599,
  'page_char_count': 751,
  'page_word_count': 122,
  'page_token_count': 187.75,
  'text': 'Learning Activities  Technology Note: The second edition of the Human  Nutrition Open Educational Resource (OER) textbook  features interactive learning activities.\xa0 These activities are  available in the web-based textbook and not available in the  downloadable versions (EPUB, Digital PDF, Print_PDF, or  Open Document).  Learning activities may be used across various mobile  devices, however, for the best user experience it is strongly  recommended that users complete these activities using a  desktop or laptop computer and in Google Chrome.  \xa0 An interactive or media element has been  excluded from this version of the text. You can  view it online here:  http://pressbooks.oer.hawaii.edu/ humannutrition2/?p=352  \xa0 The Body’s Offense  |  599',
  'sentences': ['Learning Activities  Technology Note: The second edition of the Human  Nutrition Open Educational Resourc

In [39]:
df=pd.DataFrame(pages_and_text)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_token_count,pages_sentence_count_spacy,num_chunks
count,1208.00,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,198.30,287.00,10.32,1.53
std,348.86,560.38,95.76,140.10,6.30,0.64
min,-41.00,0.00,1.00,0.00,0.00,0.00
25%,260.75,762.00,134.00,190.50,5.00,1.00
50%,562.50,1231.50,214.50,307.88,10.00,1.00
75%,864.25,1603.50,271.00,400.88,15.00,2.00
max,1166.00,2308.00,429.00,577.00,28.00,3.00


In [40]:
# spllitting each chunk intop its own item
# this makes easy working with text

# 143

In [ ]:
import re

pages_and_chunks=[]

for item in tqdm(pages_and_text):
    for sentence_chunk in item["sentences_chunks"]:
        chunk_dict={}
        chunk_dict["page_number"]=item["page_number"]

        joined_sentences_chunk="".join(sentence_chunk).replace(" "," ").strip()

        chunk_dict["sentence_chunk"]=joined_sentences_chunk

        chunk_dict["chunk_char_count"]=j
    